In [0]:
from pyspark.sql.functions import avg, count, max, min, round, current_timestamp

# Read Silver
df_silver = spark.table("healthcare_silver.wait_times_clean")

# Gold aggregations by city
df_gold = (df_silver
    .groupBy("city", "processing_date")
    .agg(
        round(avg("wait_time_hours"), 2).alias("avg_wait_hours"),
        round(max("wait_time_hours"), 2).alias("max_wait_hours"),
        round(min("wait_time_hours"), 2).alias("min_wait_hours"),
        count("hospital_id").alias("total_hospitals"),
        round(avg("patients_waiting"), 0).alias("avg_patients"),
        count(
            df_silver.wait_category == "HIGH"
        ).alias("high_wait_count")
    )
    .withColumn("gold_timestamp", current_timestamp())
)

# Write Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_gold")
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_gold.city_metrics")
)

print("✅ Gold layer complete!")
df_gold.show()

✅ Gold layer complete!
+---------+---------------+--------------+--------------+--------------+---------------+------------+---------------+--------------------+
|     city|processing_date|avg_wait_hours|max_wait_hours|min_wait_hours|total_hospitals|avg_patients|high_wait_count|      gold_timestamp|
+---------+---------------+--------------+--------------+--------------+---------------+------------+---------------+--------------------+
| Winnipeg|     2026-09-15|         12.11|         23.82|          1.04|             25|       112.0|             25|2026-09-16 03:22:...|
|Vancouver|     2026-09-15|         14.24|         23.16|          3.07|             16|       108.0|             16|2026-09-16 03:22:...|
|  Halifax|     2026-09-15|          9.19|         21.37|          1.17|             24|       111.0|             24|2026-09-16 03:22:...|
|  Toronto|     2026-09-15|         13.45|         23.85|          1.89|             22|       116.0|             22|2026-09-16 03:22:...|
|  C